# Global optimization in Python — escape local minima

Many real objectives are **smooth but non-convex**: a landscape of hills and
valleys where the best point hides among many almost-as-good ones. A plain
gradient descent from a single start slides into the
nearest local minimum and stops. **quicopt** is built to search past local
minima, and it takes the model directly in Python.

In [1]:
%pip install "quicopt[pyomo]"

## The naive approach

The common patch is to run a gradient minimizer from many random starts, or reach
for basin-hopping / a coarse grid search, and hope one run lands in the global
basin. That is expensive and still offers no coverage guarantee.

## Model it and solve

Here is Himmelblau's function — a standard non-convex test with four separate
global minima (all with value 0) — under a linear constraint, modeled in
**Pyomo** (`pip install "quicopt[pyomo]"`):

In [2]:
import pyomo.environ as pyo
from quicopt import Client
# Himmelblau: a smooth non-convex objective with four separate global minima.
m = pyo.ConcreteModel()
m.x = pyo.Var(bounds=(-5, 5))
m.y = pyo.Var(bounds=(-5, 5))
m.c = pyo.Constraint(expr=m.x + m.y <= 5)
m.obj = pyo.Objective(expr=(m.x**2 + m.y - 11)**2 + (m.x + m.y**2 - 7)**2, sense=pyo.minimize)
print(Client().solve(m).display)

▄▀▀▀▄        ▄                     █
█   █ █   █ ▄▄  ▄▀▀▀▄ ▄▀▀▀▄ █▀▀▀▄ ▀█▀▀
█ ▀▄▀ █   █  █  █   ▄ █   █ █▄▄▄▀  █  ▄
 ▀▀ ▀  ▀▀▀▀ ▀▀▀  ▀▀▀   ▀▀▀  █       ▀▀

├── shots
│   ├── 1 · primal-dual · local   0.0   0.21s  ◀ best
│   ├── 2 · primal                0.0    2.0s
│   └── 3 · primal                0.0    2.0s
├── status:     optimal
├── feasible:   true
├── objective:  1.6040127473878978e-20
├── x:          x1=-2.8051, x2=3.1313  (2 variables)
└── solve_time: 2.0008 s


## What you get

quicopt reaches a global minimum at `(-2.805, 3.131)` with objective `≈ 0` (the
`e-20` is numerical zero) — one of Himmelblau's four global optima — rather than
stalling in a nearby local basin. No restart loop, no grid.

The `shots` block tells you what that is worth. `status: optimal` means the
solver converged, and the winning shot is labelled `local`: it vouches for a
local minimum, not a global one. So read this as a strong global search that
found `0`, and not as a proof that nothing lower exists.

Non-convex objectives with dense or implicit Hessians are quicopt's structural
strength: it needs only to feel the slope, not a full second-derivative map. The
same call handles higher-dimensional, tangled landscapes where Newton-class
methods slow down.

## Next

- The problem class behind this: [Nonlinear programming (NLP)](https://quicopt.com/problems/nonconvex-nlp)
- A runnable model for every supported class: [Examples](https://quicopt.com/developer/tutorials/examples)
- Set up the client and solve your first model: [Getting started](https://quicopt.com/developer/getting-started)

**Reference:** D. M. Himmelblau, *Applied Nonlinear Programming*, McGraw-Hill,
1972.

## Frequently asked questions

### Why does a gradient-based minimizer get stuck?

Gradient methods converge to whatever local minimum is nearest the start point. On a non-convex landscape that is often not the global one, which is why people bolt on random multi-start or basin-hopping.

### Does quicopt guarantee the global optimum?

No solver guarantees the global optimum of a general non-convex problem cheaply. quicopt's quantum-inspired search is built to escape local minima and, on this problem, reaches a global minimum (objective ≈ 0). Treat it as a strong global search, not a proof.

### Is quicopt free to use?

Yes — pip install quicopt and your first call sets up a free key, no license.